# Reading a controller's response: notebook

The running example: the robot's speed loop (mass 8 kg, rolling drag 2 N·s/m, P gain Kv = 6 N per m/s, force limit 4 N) and its heading loop (yaw inertia 0.1 kg·m², PD gains, torque limit 0.2 N·m), each given a step command of 0.5 (m/s or rad). Every number in the Note is computed here. The model lives in `images/stepkit.py`, shared with the figures.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import pandas as pd
from stepkit import (M, B, I, F_MAX, TAU_MAX, V_REF, TH_REF, KP_V, KP_TH, KD_OLD, KD_NEW,
                     speed_p, heading_pd, second_order, wn_zeta, roots, step_info)
t = np.linspace(0, 60, 600001)   # 0.1 ms steps; long enough for every response to settle

## 1. The opening problem (Note, Section 1)

In [2]:
v = speed_p(t)
th_old, _, _ = heading_pd(t, kd=KD_OLD)
S = step_info(t, v, KP_V / (KP_V + B) * V_REF, V_REF)
H = step_info(t, th_old, TH_REF, TH_REF)
print('final speed', round(v[-1], 4), 'm/s; short by', round(V_REF - v[-1], 4), '=', round(100 * (V_REF - v[-1]) / V_REF, 1), '%')
print('heading peak', round(H['peak'], 4), 'rad at', round(H['t_peak'], 4), 's; overshoot', round(100 * H['overshoot'], 2), '%; settles', round(H['settle'], 2), 's')

final speed 0.375 m/s; short by 0.125 = 25.0 %
heading peak 0.7633 rad at 1.6032 s; overshoot 52.66 %; settles 9.8 s


## 2. The four numbers (Note, Section 2)

In [3]:
pd.DataFrame({k: {'speed (P, Kv = 6)': S[k], 'heading (PD, Kp = 0.4, Kd = 0.08)': H[k]}
              for k in ['t10', 't90', 'rise', 'overshoot', 'peak', 't_peak', 'settle', 'sse']}).T.round(4)

,"speed (P, Kv = 6)","heading (PD, Kp = 0.4, Kd = 0.08)"
t10,0.1054,0.2326
t90,2.3026,0.8343
rise,2.1972,0.6017
overshoot,0.0000,0.5266
peak,0.3750,0.7633
t_peak,37.4300,1.6032
settle,3.9121,9.8010
sse,0.1250,0.0000


In [4]:
print('overshoot from the peak:', (0.7633 - 0.5) / 0.5)
# Section 2.2: the force the P loop asks for, and the distance driven
u = KP_V * (V_REF - v)
x = 0.375 * (t - (1 - np.exp(-t)))
print('force at start', u[0], 'N, at the end', round(u[-1], 4), 'N; drag at 0.375 m/s', B * 0.375, 'N')
print('distance after 8 s', round(np.interp(8, t, x), 3), 'm; it grows by 0.375 m every second once the speed is steady')

overshoot from the peak: 0.5266
force at start 3.0 N, at the end 0.75 N; drag at 0.375 m/s 0.75 N
distance after 8 s 2.625 m; it grows by 0.375 m every second once the speed is steady


## 3. Error dynamics (Note, Section 3)

Check that the simulated errors satisfy the two equations.

In [5]:
e = TH_REF - th_old
de, dde = np.gradient(e, t), np.gradient(np.gradient(e, t), t)
print('heading: largest |0.1 e" + 0.08 e\' + 0.4 e| =', np.abs(I * dde + KD_OLD * de + KP_TH * e)[10:-10].max().round(8))
ev = V_REF - v
print('speed: largest |8 e\' + 8 e - 1| =', np.abs(M * np.gradient(ev, t) + (KP_V + B) * ev - B * V_REF)[10:-10].max().round(8))
print('lowest heading error', e.min().round(4), 'rad at', t[e.argmin()].round(4), 's')

heading: largest |0.1 e" + 0.08 e' + 0.4 e| = 0.0


speed: largest |8 e' + 8 e - 1| = 0.0
lowest heading error -0.2633 rad at 1.6032 s


## 4. The first-order speed loop (Note, Section 4)

In [6]:
T = M / (KP_V + B)
print('time constant T =', T, 's')
for k in [1, 2, 3, 4]:
    print(f'after {k}T: {100 * (1 - np.exp(-k)):.1f} % of the way')
print('t10 =', round(np.log(1 / 0.9), 4), ' t90 =', round(np.log(10), 4), ' rise = ln 9 =', round(np.log(9), 4), ' settle = ln 50 =', round(np.log(50), 4))
print('starting slope', 0.375 / T, 'm/s^2, force at start', KP_V * V_REF, 'N, acceleration', KP_V * V_REF / M, 'm/s^2')

time constant T = 1.0 s
after 1T: 63.2 % of the way
after 2T: 86.5 % of the way
after 3T: 95.0 % of the way
after 4T: 98.2 % of the way
t10 = 0.1054  t90 = 2.3026  rise = ln 9 = 2.1972  settle = ln 50 = 3.912
starting slope 0.375 m/s^2, force at start 3.0 N, acceleration 0.375 m/s^2


In [7]:
rows = []
for kv in [2, 6, 8, 20, 98]:
    rows.append(dict(Kv=kv, T=M / (kv + B), dc_gain=kv / (kv + B), final=kv / (kv + B) * V_REF,
                     sse=B * V_REF / (kv + B), start_force=kv * V_REF))
pd.DataFrame(rows).round(4)

,Kv,T,dc_gain,final,sse,start_force
0,2,2.0000,0.5000,0.2500,0.2500,1.0
1,6,1.0000,0.7500,0.3750,0.1250,3.0
2,8,0.8000,0.8000,0.4000,0.1000,4.0
3,20,0.3636,0.9091,0.4545,0.0455,10.0
4,98,0.0800,0.9800,0.4900,0.0100,49.0


## 5. The second-order heading loop (Note, Section 5)

In [8]:
for kd in [KD_OLD, KD_NEW, 0.28, 0.8, 0.0]:
    wn, z = wn_zeta(KP_TH, kd)
    print(f'Kd = {kd}: wn = {wn:.3f} rad/s, zeta = {z:.3f}, roots {np.round(roots(wn, z), 4)}')
print('critical Kd = 2 sqrt(Kp I) =', 2 * np.sqrt(KP_TH * I), '; for RO-060 (Kp = 0.2):', round(2 * np.sqrt(0.2 * I), 4))
wd = 2 * np.sqrt(1 - 0.2**2)
print('wd =', round(wd, 4), 'rad/s; period of the undamped swing 2 pi / 2 =', round(np.pi, 4), 's')
print('sqrt(0.64 - 16) =', np.sqrt(complex(0.64 - 16)))

Kd = 0.08: wn = 2.000 rad/s, zeta = 0.200, roots [-0.4+1.9596j -0.4-1.9596j]
Kd = 0.4: wn = 2.000 rad/s, zeta = 1.000, roots [-2.+0.j -2.+0.j]
Kd = 0.28: wn = 2.000 rad/s, zeta = 0.700, roots [-1.4+1.4283j -1.4-1.4283j]
Kd = 0.8: wn = 2.000 rad/s, zeta = 2.000, roots [-7.4641+0.j -0.5359+0.j]
Kd = 0.0: wn = 2.000 rad/s, zeta = 0.000, roots [-0.+2.j  0.-2.j]
critical Kd = 2 sqrt(Kp I) = 0.4 ; for RO-060 (Kp = 0.2): 0.2828
wd = 1.9596 rad/s; period of the undamped swing 2 pi / 2 = 3.1416 s
sqrt(0.64 - 16) = 3.919183588453085j


## 6. From zeta and wn to the numbers, and back (Note, Section 6)

In [9]:
z, wn = 0.2, 2.0
wd = wn * np.sqrt(1 - z**2)
Tp = np.pi / wd
print('Tp =', round(Tp, 4), 's; zeta wn Tp =', round(z * wn * Tp, 4), '; overshoot e^-(..) =', round(np.exp(-z * wn * Tp), 4))
print('formula e^(-pi z / sqrt(1 - z^2)) =', round(np.exp(-np.pi * z / np.sqrt(1 - z**2)), 4))
print('error at Tp:', round(-0.5 * np.exp(-z * wn * Tp), 4), 'rad')
print('zeta/sqrt(1-zeta^2) =', round(z / np.sqrt(1 - z**2), 4), '; slope check -0.4 + 1.9596*0.2041 =', round(-0.4 + wd * z / np.sqrt(1 - z**2), 6))
print('settling rule 4/(zeta wn) =', 4 / (z * wn), 's; measured', round(H['settle'], 2), 's')

Tp = 1.6032 s; zeta wn Tp = 0.6413 ; overshoot e^-(..) = 0.5266
formula e^(-pi z / sqrt(1 - z^2)) = 0.5266
error at Tp: -0.2633 rad
zeta/sqrt(1-zeta^2) = 0.2041 ; slope check -0.4 + 1.9596*0.2041 = 0.0
settling rule 4/(zeta wn) = 10.0 s; measured 9.8 s


In [10]:
from scipy.optimize import brentq
x = brentq(lambda x: (1 + x) * np.exp(-x) - 0.02, 1, 10)
print('critical damping: (1 + wn t) e^(-wn t) = 0.02 at wn t =', round(x, 3), '-> t =', round(x / 2, 3), 's at wn = 2')
print('check (1 + 5.834) e^-5.834 =', round((1 + 5.834) * np.exp(-5.834), 4), ' e^-5.834 =', round(np.exp(-5.834), 6))
for wn in [1, 2, 4]:
    i = step_info(t, second_order(t, wn, 0.2), TH_REF, TH_REF)
    print(f'wn = {wn}: overshoot {100 * i["overshoot"]:.1f} %, Tp {i["t_peak"]:.3f} s, settles {i["settle"]:.2f} s, Kp = {I * wn**2:.3f}, start torque {I * wn**2 * TH_REF:.3f} N m')

critical damping: (1 + wn t) e^(-wn t) = 0.02 at wn t = 5.834 -> t = 2.917 s at wn = 2
check (1 + 5.834) e^-5.834 = 0.02  e^-5.834 = 0.002926
wn = 1: overshoot 52.7 %, Tp 3.206 s, settles 19.60 s, Kp = 0.100, start torque 0.050 N m
wn = 2: overshoot 52.7 %, Tp 1.603 s, settles 9.80 s, Kp = 0.400, start torque 0.200 N m
wn = 4: overshoot 52.7 %, Tp 0.802 s, settles 4.90 s, Kp = 1.600, start torque 0.800 N m


In [11]:
OS = 0.05
print('zeta for 5 % overshoot:', round(-np.log(OS) / np.sqrt(np.pi**2 + np.log(OS)**2), 3))
for kd in [KD_NEW, 0.28]:
    th, w, tq = heading_pd(t, kd=kd)
    i = step_info(t, th, TH_REF, TH_REF)
    print(f'Kd = {kd}: rise {i["rise"]:.2f} s, overshoot {100 * i["overshoot"]:.1f} %, settles {i["settle"]:.2f} s, torque from {tq.min():.3f} to {tq.max():.3f} N m')
print('Kp = I wn^2 =', I * 4, '; Kd = 2 zeta wn I =', 2 * 1 * 2 * I)
th60, _, _ = heading_pd(t, kp=0.2, kd=0.283)
print('RO-060 pair (0.2, 0.283): zeta', round(wn_zeta(0.2, 0.283)[1], 4), 'settles', round(step_info(t, th60, TH_REF, TH_REF)['settle'], 2), 's')

zeta for 5 % overshoot: 0.69


Kd = 0.4: rise 1.68 s, overshoot 0.0 %, settles 2.92 s, torque from -0.027 to 0.200 N m


Kd = 0.28: rise 1.06 s, overshoot 4.6 %, settles 2.99 s, torque from -0.042 to 0.200 N m
Kp = I wn^2 = 0.4 ; Kd = 2 zeta wn I = 0.4


RO-060 pair (0.2, 0.283): zeta 1.0006 settles 4.13 s
